# Treinamento UNet Baseline em Cityscapes
Este notebook mostra como treinar o UNet (PyTorch) para segmentação usando GPU (por enquanto).

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import torch
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
from src.models.unet import UNet
#from utils.losses import DiceLoss, FocalLoss

## Carregar e preparar o dataset Cityscapes

In [ ]:
# Classe CityscapesDataset completa para uso direto
class CityscapesDataset(Dataset):
    def __init__(self, img_dir, mask_dir, img_height=96, img_width=256, num_classes=19):
        self.img_files = sorted([f for f in os.listdir(img_dir) if f.endswith('.png')])
        self.mask_files = sorted([f for f in os.listdir(mask_dir) if f.endswith('.png')])
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_height = img_height
        self.img_width = img_width
        self.num_classes = num_classes
    
    def __len__(self):
        return len(self.img_files)
    
    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.img_dir, self.img_files[idx])).convert('RGB')
        img = img.resize((self.img_width, self.img_height))
        img = np.array(img, dtype=np.float32) / 255.0
        mask = Image.open(os.path.join(self.mask_dir, self.mask_files[idx]))
        mask = mask.resize((self.img_width, self.img_height), Image.NEAREST)
        mask = np.array(mask, dtype=np.uint8)
        
        # Mapeamento RGB -> trainId
        cityscapes_color_to_train = {
            (128, 64, 128): 0, (244, 35, 232): 1, (70, 70, 70): 2, (102, 102, 156): 3,
            (190, 153, 153): 4, (153, 153, 153): 5, (250, 170, 30): 6, (220, 220, 0): 7,
            (107, 142, 35): 8, (152, 251, 152): 9, (70, 130, 180): 10, (220, 20, 60): 11,
            (255, 0, 0): 12, (0, 0, 142): 13, (0, 0, 70): 14, (0, 60, 100): 15,
            (0, 80, 100): 16, (0, 0, 230): 17, (119, 11, 32): 18
        }
        h, w = mask.shape[:2]
        trainid_mask = 255 * np.ones((h, w), dtype=np.uint8)
        palette_colors = np.array(list(cityscapes_color_to_train.keys()))
        palette_trainids = np.array(list(cityscapes_color_to_train.values()))
        mask_flat = mask.reshape(-1, 3).astype(np.int32)
        for i in range(len(mask_flat)):
            pixel = mask_flat[i]
            distances = np.sqrt(np.sum((palette_colors - pixel)**2, axis=1))
            closest_idx = np.argmin(distances)
            if distances[closest_idx] < 50:
                trainid_mask.flat[i] = palette_trainids[closest_idx]
        
        # One-hot encoding
        onehot = np.zeros((h, w, self.num_classes), dtype=np.float32)
        for c in range(self.num_classes):
            onehot[:, :, c] = (trainid_mask == c).astype(np.float32)
        img = torch.tensor(img).permute(2, 0, 1).float()
        mask_onehot = torch.tensor(onehot).float()
        return img, mask_onehot

img_dir = r'd:\Documentos\Git\edge-segmentation-lab\data\train\img'
mask_dir = r'd:\Documentos\Git\edge-segmentation-lab\data\train\label'
dataset = CityscapesDataset(img_dir, mask_dir)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)  # batch_size ajuda na velocidade do treinamento? (pesquisar)

## Inicializar o modelo UNet para GPU

In [ ]:
# Verifique se está usando GPU e qual dispositivo
print('torch.cuda.is_available():', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Nome da GPU:', torch.cuda.get_device_name(0))
    print('Memória total da GPU (MB):', torch.cuda.get_device_properties(0).total_memory // (1024*1024))
    print('Memória livre (MB):', torch.cuda.memory_allocated(0) // (1024*1024))
else:
    print('No GPU found, usando CPU')

 ## Loop de treinamento


In [ ]:
dice_loss = DiceLoss()
focal_loss = FocalLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 5 # epocas necessarias
for epoch in range(num_epochs):
    for imgs, masks in dataloader:
        imgs = imgs.to(device)
        masks = masks.permute(0, 3, 1, 2).to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = dice_loss(outputs, masks) + focal_loss(outputs, masks)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1} - Loss: {loss.item()}')

## Salvar modelo treinado

In [ ]:
torch.save(model.state_dict(), r'D:\Documentos\Git\edge-segmentation-lab\notebooks\unet_baseline.pth')

In [ ]:
# Verificando treinamdno na imagem train119.png e visualizar a máscara
import matplotlib.pyplot as plt
img_path = r'd:\Documentos\Git\edge-segmentation-lab\data\train\img\train119.png'
img = Image.open(img_path).convert('RGB')
img = img.resize((256, 96))
img_np = np.array(img, dtype=np.float32) / 255.0
img_tensor = torch.tensor(img_np).permute(2, 0, 1).unsqueeze(0).float().to(device)
model.eval()

with torch.no_grad():
    output = model(img_tensor)
    pred_mask = output.squeeze(0).cpu().numpy()
    pred_mask_argmax = np.argmax(pred_mask, axis=0)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.title('Imagem original')
plt.imshow(img)
plt.axis('off')
plt.subplot(1,2,2)
plt.title('Máscara predita (argmax)')
plt.imshow(pred_mask_argmax, cmap='tab20')
plt.axis('off')
plt.show()

In [ ]:
# Visualizar máscara predita com as cores originais do cityscapes e comparar com a máscara real
def decode_cityscapes_mask(mask_argmax):
    # Paleta Cityscapes (19 classes)
    palette = [
        (128, 64, 128), (244, 35, 232), (70, 70, 70), (102, 102, 156),
        (190, 153, 153), (153, 153, 153), (250, 170, 30), (220, 220, 0),
        (107, 142, 35), (152, 251, 152), (70, 130, 180), (220, 20, 60),
        (255, 0, 0), (0, 0, 142), (0, 0, 70), (0, 60, 100),
        (0, 80, 100), (0, 0, 230), (119, 11, 32)
    ]
    h, w = mask_argmax.shape
    mask_rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for c, color in enumerate(palette):
        mask_rgb[mask_argmax == c] = color
    return mask_rgb

# Carregar máscara real para comparação
real_mask_path = r'd:\Documentos\Git\edge-segmentation-lab\data\train\label\train119.png'
real_mask = Image.open(real_mask_path).resize((256, 96), Image.NEAREST)
real_mask_np = np.array(real_mask, dtype=np.uint8)

# Mostrar imagem, máscara predita (cores originais) e máscara real
pred_mask_rgb = decode_cityscapes_mask(pred_mask_argmax)
plt.figure(figsize=(18,5))
plt.subplot(1,3,1)
plt.title('Imagem original')
plt.imshow(img)
plt.axis('off')
plt.subplot(1,3,2)
plt.title('Máscara predita (Cityscapes)')
plt.imshow(pred_mask_rgb)
plt.axis('off')
plt.subplot(1,3,3)
plt.title('Máscara real (Cityscapes)')
plt.imshow(real_mask_np)
plt.axis('off')
plt.show()